<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/4%20Linear%20Regression/Lab%3A%20Linear%20Regression%20from%20Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression

Here’s the revised lab that incorporates each point of feedback, retaining all original sections and details while making adjustments to improve accessibility for beginners.

---

## **Lab: Building Linear Regression from Scratch**

### **Objective**
In this lab, we’ll implement linear regression step-by-step in Python. We’ll cover essential tasks, such as handling outliers, scaling features, building a `scikit-learn`-compatible model, and evaluating its performance. Each section is explained in detail to make it beginner-friendly.

### **Lab Outline**

1. **Introduction to Linear Regression**
2. **Data Generation and Initial Exploration**
3. **Handling Outliers with the IQR Rule**
4. **Feature Scaling with `StandardScaler`**
5. **Implementing Linear Regression from Scratch**
6. **Evaluating the Model**
7. **Comparison with `scikit-learn`’s LinearRegression**
8. **Optional Exploration: Overfitting and Underfitting**

---

### **1. Introduction to Linear Regression**

**Linear regression** is a basic machine learning algorithm used to model the relationship between input features (independent variables) and a target variable (dependent variable). The goal is to fit a line (or hyperplane) through the data points that best represents this relationship.

#### Real-World Analogy
Think of linear regression like predicting house prices based on square footage. We can observe a general trend: larger houses tend to be more expensive. Linear regression helps us quantify this relationship, so we can predict house prices based on square footage and other factors.

#### Mathematical Model
The linear model can be written as:
\[
y = X \theta + \epsilon
\]
where:
- \( y \) is the predicted target variable,
- \( X \) is the feature matrix (input variables),
- \( \theta \) represents the weights or coefficients for each feature,
- \( \epsilon \) is the error term, representing the difference between actual and predicted values.

---

### **2. Data Generation and Initial Exploration**

Let’s start by generating synthetic data, including some random noise to simulate real-world data. We’ll add a few extreme values (outliers) to observe their effect on the model.

```python
import numpy as np
import matplotlib.pyplot as plt

# Generate synthetic data for testing
np.random.seed(0)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

# Introduce outliers
X_outliers = np.array([[1.5], [2.0], [2.5]])
y_outliers = np.array([[10], [12], [15]])
X = np.vstack([X, X_outliers])
y = np.vstack([y, y_outliers])

# Plot the data with outliers
plt.scatter(X, y, color='blue', label="Data with Outliers")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Scatter Plot of Data with Outliers")
plt.legend()
plt.show()
```

---

### **3. Handling Outliers with the IQR Rule**

Outliers are extreme values that can distort the regression line, pulling it toward them and resulting in biased estimates. To manage this, we’ll use the **1.5x IQR (Interquartile Range) rule** to identify and remove outliers.

#### Explanation of IQR Rule
- **Step 1**: Calculate the first quartile (Q1) and third quartile (Q3) for the target variable \( y \).
- **Step 2**: Calculate the Interquartile Range (IQR), which is \( Q3 - Q1 \).
- **Step 3**: Define the lower and upper bounds for outliers as \( Q1 - 1.5 \times \text{IQR} \) and \( Q3 + 1.5 \times \text{IQR} \), respectively.
- **Step 4**: Filter out values outside these bounds.

```python
import pandas as pd

# Define a function to remove outliers using the 1.5x IQR rule
def remove_outliers_iqr(X, y):
    # Convert y to a DataFrame for easy manipulation with pandas
    y_df = pd.DataFrame(y, columns=['target'])
    
    # Step 1: Calculate Q1 (25th percentile) and Q3 (75th percentile) for y
    Q1 = y_df['target'].quantile(0.25)
    Q3 = y_df['target'].quantile(0.75)
    
    # Step 2: Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1
    
    # Step 3: Define the lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Step 4: Filter out the outliers
    mask = (y_df['target'] >= lower_bound) & (y_df['target'] <= upper_bound)
    X_filtered = X[mask.values]  # mask.values to apply filter to numpy array X
    y_filtered = y[mask.values]
    
    return X_filtered, y_filtered

# Remove outliers and plot the updated data
X, y = remove_outliers_iqr(X, y)
plt.scatter(X, y, color='green', label="Data without Outliers")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Scatter Plot after Removing Outliers")
plt.legend()
plt.show()
```

---

### **4. Feature Scaling with `StandardScaler`**

Feature scaling ensures that all features are on a comparable scale, which helps the model converge faster and makes it more interpretable. We’ll use `StandardScaler` from `scikit-learn` to standardize the features, so they have a mean of 0 and a standard deviation of 1.

```python
from sklearn.preprocessing import StandardScaler

# Standardize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

---

### **5. Implementing Linear Regression from Scratch**

Now, we’ll create a custom `MyLinearRegressor` class, following `scikit-learn` conventions, and implementing the **Normal Equation** for solving linear regression directly. This involves calculating weights using matrix operations.

#### Explanation of the Class Methods
- **`fit` Method**: Calculates the weights using the Normal Equation. This approach doesn’t require iterative optimization.
- **`predict` Method**: Generates predictions using these weights.
- **`score` Method**: Calculates the \( R^2 \) score, a common metric to evaluate model fit.

```python
from sklearn.base import BaseEstimator, RegressorMixin

class MyLinearRegressor(BaseEstimator, RegressorMixin):
    def __init__(self):
        self.theta = None  # To store weights after training
    
    def fit(self, X, y):
        # Add a bias term (column of ones) to X
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        # Calculate theta using the normal equation:
        # theta = (X_b.T @ X_b)^-1 @ X_b.T @ y
        self.theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        return self
    
    def predict(self, X):
        # Add bias term to X and compute predictions
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta
    
    def score(self, X, y):
        # Calculate R^2 score
        y_pred = self.predict(X)
        u = ((y - y_pred) ** 2).sum()
        v = ((y - y.mean()) ** 2).sum()
        return 1 - u / v
```

**Note**: `BaseEstimator` and `RegressorMixin` make this model compatible with `scikit-learn` utilities, such as `cross_val_score`, which allows us to validate model performance more easily.

---

### **6. Evaluating the Model**

We’ll evaluate the model using **Mean Squared Error (MSE)** and \( R^2 \) score.

- **MSE**: Measures the average squared difference between actual and predicted values.
- **\( R^2 \)**: Indicates the proportion of the variance in the target variable explained by the model.

```python
# Initialize and fit the model
model = MyLinearRegressor()
model.fit(X_scaled, y)

# Predict values and evaluate
y_pred = model.predict(X_scaled)
mse = np.mean((y - y_pred) ** 2)
print("Mean Squared Error:", mse)

r2_score = model.score(X_scaled, y)
print("R^2 Score:", r2_score)
```

---

### **7. Comparison with `scikit-learn`’s LinearRegression**

To validate our model, we’ll compare it with `scikit-learn`’s `LinearRegression` and use cross-validation to check its reliability.

```python
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Initialize and fit scikit-learn's LinearRegression
sklearn_model = LinearRegression()
scores = cross_val_score(sklearn_model, X_scaled, y, cv=5, scoring='r2')
print("Cross-validated R^2 scores (scikit-learn):", scores)
print("Mean R^2 score (scikit-learn):", scores.mean())
```

---

### **8. Optional Exploration: Overfitting and Underfitting**

To explore overfitting, add polynomial features and observe changes in performance metrics.## Objective:
In this hands-on lab, you will:
- Load and explore the **California Housing Dataset** for regression tasks.
- Implement **Linear Regression** using `scikit-learn`.
- Train, evaluate, and visualize the model's performance.
- Learn key concepts such as feature scaling and performance metrics (e.g., MSE).

## 1. **Loading Libraries and the Dataset**

#### Instructions:
We will start by importing the necessary libraries. `numpy` is for numerical operations, `pandas` for handling datasets, and `scikit-learn` provides the tools for building the Linear Regression model.

In [ ]:
# Step 1: Import the required libraries

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

Now, let's load the **California Housing Dataset**, which contains data such as average house prices and features like the number of rooms, income, and location.

In [ ]:
# Step 2: Load the dataset

data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # Median house value

## 2. **Exploring the Dataset**

#### Instructions:
Before building the model, let’s explore the dataset to understand its structure and the distribution of the features.

#### Task:
- Display the first few rows of the dataset.
- Check the shape of the data (number of samples and features).
- Summarize the statistics of the features using `describe()`.

In [ ]:
# Step 3: Explore the dataset

# Task: Display the first few rows of the data
# Insert your code here:
# Example: print(X.head())

# Task: Check the shape of the data
# Insert your code here:
# Example: print(X.shape)

# Task: Describe the summary statistics of the features
# Insert your code here:
# Example: print(X.describe())

#### Reflection Prompt:
- Why do we load the target variable (`y`) separately from the features (`X`)?
- What kind of features do you think will have the most impact on house prices?


## 3. **Splitting the Data**

#### Instructions:
Now, we will split the dataset into **training** and **test** sets. The training set will be used to train the model, while the test set will evaluate the model's performance on unseen data.

#### Task:
- Use `train_test_split()` to split the dataset into 80% training data and 20% test data.

In [ ]:
# Step 4: Split the data into training and test sets

# Task: Split the data into 80% training and 20% test sets
# Insert your code here:
# Example: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Reflection Prompt:
- Why is it important to set aside a portion of the data for testing?
- What might happen if you evaluate the model only on the training set?

## 4. **Standardizing the Data**

#### Instructions:
Linear regression models can be sensitive to the scale of the features. Therefore, it’s important to standardize the features so they all contribute equally.

#### Task:
- Initialize a `StandardScaler` object.
- Fit the scaler to the training data.
- Use the scaler to transform the training and test sets.

In [ ]:
# Step 5: Standardize the feature data using StandardScaler

# Task: Initialize the scaler
scaler = StandardScaler()

# Task: Fit the scaler on the training data
# Insert your code here:
# Example: scaler.fit(X_train)

# Task: Transform the training and test data
# Insert your code here:
# Example: X_train_scaled = scaler.transform(X_train)
# Example: X_test_scaled = scaler.transform(X_test)

# Task: Check the transformed data by displaying the first 5 rows
# Insert your code here:
# Example: print(X_train_scaled[:5])

#### Reflection Prompt:
- Why is it important to **fit** the scaler only on the training data before transforming the test data?
- How do you expect the scaling to affect the coefficients in the linear regression model?

## 5. **Training the Linear Regression Model**

#### Instructions:
Now we will train the **Linear Regression** model using the scaled training data.

#### Task:
- Initialize the `LinearRegression` model.
- Fit the model to the training data.
- Display the learned coefficients (weights) of the model.

In [ ]:
# Step 6: Train the Linear Regression model

# Task: Initialize the Linear Regression model
# Insert your code here:
# Example: model = LinearRegression()

# Task: Fit the model on the scaled training data
# Insert your code here:
# Example: model.fit(X_train_scaled, y_train)

# Task: Print the coefficients of the model
# Insert your code here:
# Example: print(model.coef_)

#### Reflection Prompt:
- What do the coefficients represent in the context of house prices?
- How can the intercept and coefficients help interpret the model's predictions?

## 6. **Evaluating the Model on Test Data**

#### Instructions:
After training the model, we will evaluate its performance on the **test set** using **Mean Squared Error (MSE)**. Lower MSE indicates that the model's predictions are closer to the actual values.

#### Task:
- Predict the target values on the test set.
- Calculate the Mean Squared Error between the actual and predicted values.
- Print the MSE.


In [ ]:
# Step 7: Make predictions on the test set and calculate MSE

# Task: Make predictions on the scaled test data
# Insert your code here:
# Example: y_test_pred = model.predict(X_test_scaled)

# Task: Calculate the Mean Squared Error (MSE)
# Insert your code here:
# Example: mse = mean_squared_error(y_test, y_test_pred)

# Task: Print the MSE value
# Insert your code here:
# Example: print(f"Mean Squared Error on the test set: {mse}")

#### Reflection Prompt:
- Why is Mean Squared Error a useful metric for regression tasks?
- How would you explain the MSE value in terms of predicting house prices?

## 7. **Visualizing Predictions vs Actual Values**

#### Instructions:
To better understand how well the model performs, we will visualize the predicted house prices against the actual house prices.

#### Task:
- Create a scatter plot to visualize the relationship between the predicted and actual values.

In [ ]:
# Step 8: Visualize the predicted vs. actual values

# Task: Create a scatter plot
# Insert your code here:
# Example: plt.scatter(y_test, y_test_pred)
# Example: plt.xlabel("Actual Prices")
# Example: plt.ylabel("Predicted Prices")
# Example: plt.title("Actual vs. Predicted House Prices")
# Example: plt.show()

#### Reflection Prompt:
- If the predictions are perfect, where should the points lie on the scatter plot?
- What might be causing any deviations between the predicted and actual values?

## 8. **Summary and Reflection**

In this lab, you:
- Loaded and explored the **California Housing Dataset**.
- Standardized the data to improve model performance.
- Trained and evaluated a **Linear Regression** model.
- Visualized the relationship between predicted and actual values.

#### Key Takeaways:
- Linear regression helps quantify the relationship between features and a target variable.
- Feature scaling is crucial for improving the performance of models sensitive to feature magnitudes.
- Evaluating models on unseen data is essential to ensure that they generalize well.

#### Reflection Prompt:
- What are the main factors that influence the performance of a Linear Regression model?
- How might you improve the model’s performance further?